# Basic Portfolio Analysis Notebook

## Introduction

### Welcome to the Portfolio Analysis Tutorial!

Are you an investor looking to gain deeper insights into your portfolio's performance? Perhaps you're curious about how different assets contribute to your overall returns, or you want to simulate future performance to make more informed decisions. This tutorial is designed to help you achieve all that and more, even if you're not a programming expert.

In this tutorial, we'll guide you step-by-step through creating and using a Python-based portfolio analysis tool. You'll learn how to fetch financial data, calculate important performance metrics, visualize your portfolio's returns, and even run Monte Carlo simulations to forecast future performance.

**What You'll Learn:**

1. **Fetching Financial Data**: Use Python to download historical data for your favorite stocks.
2. **Calculating Performance Metrics**: Understand how to compute annual returns, volatility, Sharpe ratio, and more.
3. **Visualizing Performance**: Create beautiful plots to visualize both individual asset performance and the overall portfolio return.
4. **Monte Carlo Simulations**: Run simulations to predict how your portfolio might perform under different scenarios.

By the end of this tutorial, you'll have a powerful toolkit at your disposal, enabling you to analyze and visualize your investment portfolio with ease. Whether you're a seasoned investor or just getting started, this tutorial will provide valuable insights into the world of portfolio analysis.

Let's get started on your journey to becoming a more informed and confident investor!

## Install Required Packages

First, ensure you have the required packages installed. Run the following command in a Jupyter notebook cell:

In [ ]:
!pip install pandas yfinance numpy matplotlib -q --disable-pip-version-check

## Import Necessary Libraries

Import the necessary libraries for our analysis.

In [ ]:
import pandas as pd
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt

import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from my_portfolio import *

from datetime import datetime, timedelta


## DataLoader Class

Create a class to fetch and preprocess financial data.

In [ ]:
class DataLoader:
    def __init__(self, tickers, start_date, end_date):
        self.tickers = tickers
        self.start_date = start_date
        self.end_date = end_date

    def fetch_data(self):
        data = yf.download(self.tickers, start=self.start_date, end=self.end_date, auto_adjust=True)['Close']
        return data

## PerformanceMetrics Class

Create a class to calculate various performance metrics.

In [ ]:
class PerformanceMetrics:
    @staticmethod
    def calculate_annual_return(data):
        annual_return = data.resample('YE').last().pct_change().mean()
        return annual_return

    @staticmethod
    def calculate_annual_volatility(data):
        annual_volatility = data.pct_change().std() * (252 ** 0.5)
        return annual_volatility

    @staticmethod
    def calculate_sharpe_ratio(data, risk_free_rate=risk_free_rate):
        annual_return = PerformanceMetrics.calculate_annual_return(data)
        annual_volatility = PerformanceMetrics.calculate_annual_volatility(data)
        sharpe_ratio = (annual_return - risk_free_rate) / annual_volatility
        return sharpe_ratio

    @staticmethod
    def calculate_max_drawdown(data):
        cumulative_returns = (1 + data.pct_change(fill_method=None)).cumprod()
        peak = cumulative_returns.expanding(min_periods=1).max()
        drawdown = (cumulative_returns / peak) - 1
        max_drawdown = drawdown.min()
        return max_drawdown

    @staticmethod
    def calculate_sortino_ratio(data, risk_free_rate=risk_free_rate):
        returns = data.pct_change(fill_method=None).dropna(how='all').fillna(0)
        downside_deviation = returns[returns < 0].std() * (252 ** 0.5)
        annual_return = PerformanceMetrics.calculate_annual_return(data)
        sortino_ratio = (annual_return - risk_free_rate) / downside_deviation
        return sortino_ratio

    @staticmethod
    def calculate_var(data, confidence_level=0.95):
        returns = data.pct_change(fill_method=None).dropna(how='all').fillna(0)
        var = np.percentile(returns, (1 - confidence_level) * 100)
        return var

## PortfolioAnalysis Class

Create a class to perform portfolio analysis.

In [ ]:
class PortfolioAnalysis:
    def __init__(self, data, weights):
        self.data = data
        self.weights = np.array(weights)

        assert len(weights) == len(data.columns), "Weights must be the same length as the number of columns in the data"
        assert sum(weights) == 1, "Weights must sum to 1"

    def calculate_portfolio_return(self):
        returns = self.data.pct_change().mean()
        portfolio_return = np.dot(self.weights, returns) * 252
        return portfolio_return

    def calculate_portfolio_volatility(self):
        returns = self.data.pct_change()
        covariance_matrix = returns.cov() * 252
        portfolio_volatility = np.sqrt(np.dot(self.weights.T, np.dot(covariance_matrix, self.weights)))
        return portfolio_volatility

    def calculate_portfolio_sharpe_ratio(self, risk_free_rate=risk_free_rate):
        portfolio_return = self.calculate_portfolio_return()
        portfolio_volatility = self.calculate_portfolio_volatility()
        sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_volatility
        return sharpe_ratio

## PortfolioVisualization Class

Create a class to visualize portfolio performance, including plotting the total portfolio return over time.

In [ ]:
class PortfolioVisualization:
    @staticmethod
    def plot_performance(data):
        cumulative_returns = (1 + data.pct_change(fill_method=None)).cumprod()
        cumulative_returns.plot(figsize=(10, 6))
        plt.title('Portfolio Component Cumulative Returns')
        plt.xlabel('Date')
        plt.ylabel('Cumulative Returns')
        plt.grid(True)
        plt.show()

    @staticmethod
    def plot_portfolio_return(data, weights):
        returns = data.pct_change(fill_method=None).dropna(how='all').fillna(0)
        weighted_returns = returns.dot(weights)
        cumulative_portfolio_returns = (1 + weighted_returns).cumprod()
        cumulative_portfolio_returns.plot(figsize=(10, 6))
        plt.title('Total Portfolio Cumulative Return')
        plt.xlabel('Date')
        plt.ylabel('Cumulative Returns')
        plt.grid(True)
        plt.show()

## BenchmarkComparison Class

Compare your portfolio's performance against common market benchmarks. Calculate alpha, beta, tracking error, and information ratio.

In [ ]:
class BenchmarkComparison:
    """
    Compare portfolio performance against market benchmarks.
    
    Calculates alpha, beta, tracking error, information ratio, and
    generates comparison reports and visualizations.
    """
    
    # Common benchmarks
    BENCHMARKS = {
        'SPY': 'S&P 500 (US Large Cap)',
        'VTI': 'Total US Stock Market',
        'BND': 'Total US Bond Market',
        'VT': 'Total World Stock Market',
        'QQQ': 'NASDAQ 100',
        'IWM': 'Russell 2000 (US Small Cap)',
        'EFA': 'Developed Markets ex-US',
        'AGG': 'US Aggregate Bond'
    }
    
    def __init__(self, portfolio_data, weights, benchmark_ticker='SPY', risk_free_rate=0.02):
        """
        Initialize benchmark comparison.
        
        Parameters:
        -----------
        portfolio_data : pd.DataFrame
            Historical price data for portfolio assets
        weights : array-like
            Portfolio weights (must sum to 1.0)
        benchmark_ticker : str
            Ticker symbol for benchmark (default: 'SPY')
        risk_free_rate : float
            Annual risk-free rate for calculations (default: 0.02)
        """
        self.portfolio_data = portfolio_data
        self.weights = np.array(weights)
        self.benchmark_ticker = benchmark_ticker
        self.risk_free_rate = risk_free_rate
        
        # Calculate portfolio returns
        returns = portfolio_data.pct_change().dropna()
        self.portfolio_returns = returns.dot(self.weights)
        
        # Fetch benchmark data
        self.benchmark_data = self._fetch_benchmark()
        self.benchmark_returns = self.benchmark_data.pct_change().dropna()
        
        # Align dates
        common_dates = self.portfolio_returns.index.intersection(self.benchmark_returns.index)
        self.portfolio_returns = self.portfolio_returns.loc[common_dates]
        self.benchmark_returns = self.benchmark_returns.loc[common_dates]
    
    def _fetch_benchmark(self):
        """Fetch benchmark price data for the same date range as portfolio."""
        start_date = self.portfolio_data.index.min()
        end_date = self.portfolio_data.index.max()
        
        benchmark = yf.download(self.benchmark_ticker, start=start_date, end=end_date, progress=False, auto_adjust=True)['Close'].squeeze()
        return benchmark
    
    def calculate_beta(self):
        """
        Calculate portfolio beta relative to benchmark.
        
        Beta measures the portfolio's sensitivity to benchmark movements.
        Beta > 1 means more volatile than benchmark.
        """
        covariance = np.cov(self.portfolio_returns, self.benchmark_returns)[0, 1]
        benchmark_variance = np.var(self.benchmark_returns)
        beta = covariance / benchmark_variance
        return beta
    
    def calculate_alpha(self, annualized=True):
        """
        Calculate Jensen's alpha (CAPM alpha).
        
        Alpha measures excess return above what CAPM predicts.
        Positive alpha indicates outperformance.
        
        Parameters:
        -----------
        annualized : bool
            Whether to annualize the alpha (default: True)
        """
        beta = self.calculate_beta()
        
        portfolio_mean = self.portfolio_returns.mean()
        benchmark_mean = self.benchmark_returns.mean()
        rf_daily = self.risk_free_rate / 252
        
        # Jensen's Alpha: Rp - [Rf + Beta * (Rm - Rf)]
        alpha = portfolio_mean - (rf_daily + beta * (benchmark_mean - rf_daily))
        
        if annualized:
            alpha = alpha * 252
        
        return alpha
    
    def calculate_tracking_error(self, annualized=True):
        """
        Calculate tracking error (active risk).
        
        Tracking error is the standard deviation of the difference
        between portfolio and benchmark returns.
        
        Parameters:
        -----------
        annualized : bool
            Whether to annualize (default: True)
        """
        active_returns = self.portfolio_returns - self.benchmark_returns
        tracking_error = active_returns.std()
        
        if annualized:
            tracking_error = tracking_error * np.sqrt(252)
        
        return tracking_error
    
    def calculate_information_ratio(self):
        """
        Calculate information ratio.
        
        IR = (Portfolio Return - Benchmark Return) / Tracking Error
        
        Measures risk-adjusted excess return relative to benchmark.
        Higher is better; >0.5 is good, >1.0 is excellent.
        """
        active_return = (self.portfolio_returns.mean() - self.benchmark_returns.mean()) * 252
        tracking_error = self.calculate_tracking_error(annualized=True)
        
        if tracking_error == 0:
            return np.inf if active_return > 0 else -np.inf
        
        return active_return / tracking_error
    
    def calculate_correlation(self):
        """Calculate correlation between portfolio and benchmark returns."""
        return np.corrcoef(self.portfolio_returns, self.benchmark_returns)[0, 1]
    
    def calculate_r_squared(self):
        """
        Calculate R-squared (coefficient of determination).
        
        Measures how much of portfolio variance is explained by benchmark.
        """
        correlation = self.calculate_correlation()
        return correlation ** 2
    
    def calculate_up_capture(self):
        """
        Calculate upside capture ratio.
        
        Measures portfolio performance when benchmark is positive.
        >100% means outperforms in up markets.
        """
        up_mask = self.benchmark_returns > 0
        if up_mask.sum() == 0:
            return np.nan
        
        portfolio_up = self.portfolio_returns[up_mask].mean()
        benchmark_up = self.benchmark_returns[up_mask].mean()
        
        return (portfolio_up / benchmark_up) * 100
    
    def calculate_down_capture(self):
        """
        Calculate downside capture ratio.
        
        Measures portfolio performance when benchmark is negative.
        <100% means protects better in down markets.
        """
        down_mask = self.benchmark_returns < 0
        if down_mask.sum() == 0:
            return np.nan
        
        portfolio_down = self.portfolio_returns[down_mask].mean()
        benchmark_down = self.benchmark_returns[down_mask].mean()
        
        return (portfolio_down / benchmark_down) * 100
    
    def generate_report(self):
        """Generate a comprehensive comparison report."""
        print("\n" + "="*60)
        print("BENCHMARK COMPARISON REPORT")
        print("="*60)
        print(f"Benchmark: {self.benchmark_ticker}", end="")
        if self.benchmark_ticker in self.BENCHMARKS:
            print(f" ({self.BENCHMARKS[self.benchmark_ticker]})")
        else:
            print()
        print(f"Period: {self.portfolio_returns.index.min().strftime('%Y-%m-%d')} to {self.portfolio_returns.index.max().strftime('%Y-%m-%d')}")
        print("-"*60)
        
        # Returns
        port_annual_return = self.portfolio_returns.mean() * 252 * 100
        bench_annual_return = self.benchmark_returns.mean() * 252 * 100
        print("\nAnnualized Returns:")
        print(f"  Portfolio:  {port_annual_return:.2f}%")
        print(f"  Benchmark:  {bench_annual_return:.2f}%")
        print(f"  Difference: {port_annual_return - bench_annual_return:+.2f}%")
        
        # Volatility
        port_vol = self.portfolio_returns.std() * np.sqrt(252) * 100
        bench_vol = self.benchmark_returns.std() * np.sqrt(252) * 100
        print("\nAnnualized Volatility:")
        print(f"  Portfolio:  {port_vol:.2f}%")
        print(f"  Benchmark:  {bench_vol:.2f}%")
        
        # Risk metrics
        print("\nRisk Metrics:")
        print(f"  Beta:              {self.calculate_beta():.3f}")
        print(f"  Alpha (annual):    {self.calculate_alpha()*100:.2f}%")
        print(f"  R-squared:         {self.calculate_r_squared():.3f}")
        print(f"  Correlation:       {self.calculate_correlation():.3f}")
        
        # Performance metrics
        print("\nPerformance Metrics:")
        print(f"  Tracking Error:    {self.calculate_tracking_error()*100:.2f}%")
        print(f"  Information Ratio: {self.calculate_information_ratio():.3f}")
        print(f"  Up Capture:        {self.calculate_up_capture():.1f}%")
        print(f"  Down Capture:      {self.calculate_down_capture():.1f}%")
        print("="*60)
    
    def plot_cumulative_returns(self, initial_value=10000):
        """
        Plot cumulative returns comparison.
        
        Parameters:
        -----------
        initial_value : float
            Starting investment value for comparison
        """
        # Calculate cumulative returns
        portfolio_cum = (1 + self.portfolio_returns).cumprod() * initial_value
        benchmark_cum = (1 + self.benchmark_returns).cumprod() * initial_value
        
        plt.figure(figsize=(12, 6))
        
        plt.plot(portfolio_cum.index, portfolio_cum.values, 
                label='Portfolio', linewidth=2, color='blue')
        plt.plot(benchmark_cum.index, benchmark_cum.values, 
                label=f'Benchmark ({self.benchmark_ticker})', linewidth=2, color='orange')
        
        plt.title('Cumulative Returns: Portfolio vs Benchmark')
        plt.xlabel('Date')
        plt.ylabel(f'Value (starting from ${initial_value:,.0f})')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Add final values annotation
        final_port = portfolio_cum.iloc[-1]
        final_bench = benchmark_cum.iloc[-1]
        plt.annotate(f'${final_port:,.0f}', 
                    xy=(portfolio_cum.index[-1], final_port),
                    xytext=(10, 10), textcoords='offset points',
                    fontsize=9, color='blue')
        plt.annotate(f'${final_bench:,.0f}', 
                    xy=(benchmark_cum.index[-1], final_bench),
                    xytext=(10, -15), textcoords='offset points',
                    fontsize=9, color='orange')
        
        plt.tight_layout()
        plt.show()
    
    def plot_rolling_metrics(self, window=252):
        """
        Plot rolling alpha and beta.
        
        Parameters:
        -----------
        window : int
            Rolling window in trading days (default: 252 = 1 year)
        """
        fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
        
        # Calculate rolling metrics
        rolling_beta = []
        rolling_alpha = []
        dates = []
        
        for i in range(window, len(self.portfolio_returns)):
            port_window = self.portfolio_returns.iloc[i-window:i]
            bench_window = self.benchmark_returns.iloc[i-window:i]
            
            # Beta
            cov = np.cov(port_window, bench_window)[0, 1]
            var = np.var(bench_window)
            beta = cov / var
            
            # Alpha (annualized)
            rf_daily = self.risk_free_rate / 252
            alpha = (port_window.mean() - (rf_daily + beta * (bench_window.mean() - rf_daily))) * 252
            
            rolling_beta.append(beta)
            rolling_alpha.append(alpha)
            dates.append(self.portfolio_returns.index[i])
        
        # Plot beta
        axes[0].plot(dates, rolling_beta, linewidth=1.5, color='blue')
        axes[0].axhline(y=1.0, color='red', linestyle='--', linewidth=1, label='Beta = 1.0')
        axes[0].set_ylabel('Beta')
        axes[0].set_title(f'Rolling {window}-Day Beta and Alpha')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Plot alpha
        axes[1].plot(dates, [a * 100 for a in rolling_alpha], linewidth=1.5, color='green')
        axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1, label='Alpha = 0')
        axes[1].set_ylabel('Alpha (%)')
        axes[1].set_xlabel('Date')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

## Example Usage: Compute Portfolio Returns

Now, let's use the classes we've created to analyze a sample portfolio.

In [ ]:
# Load data
data_loader = DataLoader(tickers, start_date, end_date)
data = data_loader.fetch_data()

# Calculate performance metrics
annual_return = PerformanceMetrics.calculate_annual_return(data)
annual_volatility = PerformanceMetrics.calculate_annual_volatility(data)
sharpe_ratio = PerformanceMetrics.calculate_sharpe_ratio(data)
max_drawdown = PerformanceMetrics.calculate_max_drawdown(data)
sortino_ratio = PerformanceMetrics.calculate_sortino_ratio(data)
var = PerformanceMetrics.calculate_var(data)

print('\n********** Individual Ticker Stats **********')
print("\nAnnual Return (%):", np.round(100*annual_return, 2))
print("\nAnnual Volatility (%):", np.round(100*annual_volatility, 2))
print("\nSharpe Ratio:", np.round(sharpe_ratio, 2))
print("\nMax Drawdown (%):", np.round(100*max_drawdown, 2))
print("\nSortino Ratio:", np.round(sortino_ratio, 2))
print("\nValue at Risk (VaR):", np.round(var, 2))

# Perform portfolio analysis
portfolio_analysis = PortfolioAnalysis(data, weights)
portfolio_return = portfolio_analysis.calculate_portfolio_return()
portfolio_volatility = portfolio_analysis.calculate_portfolio_volatility()
portfolio_sharpe_ratio = portfolio_analysis.calculate_portfolio_sharpe_ratio()

print('\n********** Portfolio Return Stats **********')
print("Portfolio Return (%):", np.round(100*portfolio_return, 2))
print("Portfolio Volatility (%):", np.round(100*portfolio_volatility, 2))
print("Portfolio Sharpe Ratio:", np.round(portfolio_sharpe_ratio, 2))

## Visualize Portfolio Performance

In [ ]:
# Visualize portfolio performance
PortfolioVisualization.plot_performance(data)

In [ ]:
# Visualize total portfolio return
PortfolioVisualization.plot_portfolio_return(data, weights)

## Benchmark Comparison

Compare your portfolio against a market benchmark like the S&P 500 (SPY).

In [ ]:
# Compare portfolio against S&P 500 benchmark
benchmark = BenchmarkComparison(data, weights, benchmark_ticker='SPY', risk_free_rate=risk_free_rate)
benchmark.generate_report()
benchmark.plot_cumulative_returns()

## Monte Carlo Simulation

The Monte Carlo simulation projects potential future portfolio values by generating thousands of random return scenarios based on historical return distributions and asset correlations.

**Key Features:**
- Uses portfolio weights to calculate weighted returns
- Tracks cumulative portfolio value over time
- Provides percentile-based statistics (5th, 25th, 50th, 75th, 95th)
- Visualizes uncertainty with percentile bands
- Calculates probability of loss


### MonteCarloSimulation Class

Create a class to perform Monte Carlo simulations.

In [ ]:
class MonteCarloSimulation:
    """
    Monte Carlo simulation for portfolio performance projection.
    
    Simulates multiple future paths for a portfolio based on historical
    return distributions, accounting for asset correlations.
    """
    
    def __init__(self, data, weights, num_simulations=1000, time_horizon=252, initial_investment=10000):
        """
        Initialize Monte Carlo simulation.
        
        Parameters:
        -----------
        data : pd.DataFrame
            Historical price data for portfolio assets
        weights : array-like
            Portfolio weights (must sum to 1.0)
        num_simulations : int
            Number of simulation paths to generate
        time_horizon : int
            Number of trading days to simulate (252 = 1 year)
        initial_investment : float
            Starting portfolio value
        """
        self.data = data
        self.weights = np.array(weights)
        self.num_simulations = num_simulations
        self.time_horizon = time_horizon
        self.initial_investment = initial_investment
        
        # Validate weights
        assert len(self.weights) == len(data.columns), "Weights must match number of assets"
        assert np.isclose(sum(self.weights), 1.0), "Weights must sum to 1.0"
        
        # Store simulation results
        self._results = None

    def simulate(self):
        """
        Run Monte Carlo simulation.
        
        Returns:
        --------
        np.ndarray
            Array of shape (num_simulations, time_horizon) containing
            portfolio values for each simulation path over time.
        """
        returns = self.data.pct_change(fill_method=None).dropna()
        mean_returns = returns.mean().values
        cov_matrix = returns.cov().values

        results = np.zeros((self.num_simulations, self.time_horizon))
        
        for i in range(self.num_simulations):
            # Generate correlated random returns for all assets
            sim_returns = np.random.multivariate_normal(mean_returns, cov_matrix, self.time_horizon)
            
            # Calculate weighted portfolio returns at each time step
            portfolio_returns = sim_returns @ self.weights
            
            # Track cumulative portfolio value over time
            cumulative_returns = np.cumprod(1 + portfolio_returns)
            results[i, :] = self.initial_investment * cumulative_returns
        
        self._results = results
        return results

    def get_statistics(self, percentiles=[5, 25, 50, 75, 95]):
        """
        Calculate statistics across all simulation paths.
        
        Parameters:
        -----------
        percentiles : list
            Percentiles to calculate (default: 5, 25, 50, 75, 95)
            
        Returns:
        --------
        dict
            Dictionary containing:
            - 'percentiles': dict of percentile values at each time step
            - 'mean': mean portfolio value at each time step
            - 'std': standard deviation at each time step
            - 'final_values': summary statistics for final portfolio values
        """
        if self._results is None:
            self.simulate()
        
        results = self._results
        final_values = results[:, -1]
        
        stats = {
            'percentiles': {p: np.percentile(results, p, axis=0) for p in percentiles},
            'mean': np.mean(results, axis=0),
            'std': np.std(results, axis=0),
            'final_values': {
                'mean': np.mean(final_values),
                'median': np.median(final_values),
                'std': np.std(final_values),
                'min': np.min(final_values),
                'max': np.max(final_values),
                'percentile_5': np.percentile(final_values, 5),
                'percentile_95': np.percentile(final_values, 95),
                'prob_loss': np.mean(final_values < self.initial_investment) * 100
            }
        }
        return stats

    def plot_simulation(self, show_percentiles=True, num_paths=100):
        """
        Plot Monte Carlo simulation results with percentile bands.
        
        Parameters:
        -----------
        show_percentiles : bool
            Whether to show percentile bands (5th, 50th, 95th)
        num_paths : int
            Number of individual paths to plot (for visual clarity)
        """
        if self._results is None:
            self.simulate()
        
        results = self._results
        stats = self.get_statistics()
        
        plt.figure(figsize=(12, 7))
        
        # Plot a subset of individual paths (semi-transparent)
        paths_to_plot = min(num_paths, self.num_simulations)
        for i in range(paths_to_plot):
            plt.plot(results[i, :], color='lightblue', alpha=0.3, linewidth=0.5)
        
        if show_percentiles:
            days = np.arange(self.time_horizon)
            
            # Plot percentile bands
            plt.fill_between(days, 
                           stats['percentiles'][5], 
                           stats['percentiles'][95],
                           color='blue', alpha=0.2, label='5th-95th percentile')
            plt.fill_between(days,
                           stats['percentiles'][25],
                           stats['percentiles'][75],
                           color='blue', alpha=0.3, label='25th-75th percentile')
            
            # Plot median
            plt.plot(stats['percentiles'][50], color='darkblue', 
                    linewidth=2, label='Median (50th percentile)')
        
        # Plot initial investment line
        plt.axhline(y=self.initial_investment, color='red', 
                   linestyle='--', linewidth=1, label=f'Initial: ${self.initial_investment:,.0f}')
        
        plt.title(f'Monte Carlo Simulation ({self.num_simulations:,} paths, {self.time_horizon} days)')
        plt.xlabel('Trading Days')
        plt.ylabel('Portfolio Value ($)')
        plt.legend(loc='upper left')
        plt.grid(True, alpha=0.3)
        
        # Add summary statistics text box
        final_stats = stats['final_values']
        textstr = f"Final Value Statistics:\n"
        textstr += f"Median: ${final_stats['median']:,.0f}\n"
        textstr += f"5th %%: ${final_stats['percentile_5']:,.0f}\n"
        textstr += f"95th %%: ${final_stats['percentile_95']:,.0f}\n"
        textstr += f"Prob. of Loss: {final_stats['prob_loss']:.1f}%"
        
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        plt.text(0.98, 0.02, textstr, transform=plt.gca().transAxes, 
                fontsize=9, verticalalignment='bottom', horizontalalignment='right',
                bbox=props)
        
        plt.tight_layout()
        plt.show()
        
    def print_summary(self):
        """Print a summary of simulation results."""
        if self._results is None:
            self.simulate()
            
        stats = self.get_statistics()
        final = stats['final_values']
        
        print("\n" + "="*50)
        print("MONTE CARLO SIMULATION SUMMARY")
        print("="*50)
        print(f"Initial Investment: ${self.initial_investment:,.0f}")
        print(f"Time Horizon: {self.time_horizon} trading days")
        print(f"Number of Simulations: {self.num_simulations:,}")
        print("-"*50)
        print("Final Portfolio Value Statistics:")
        print(f"  Mean:     ${final['mean']:,.0f}")
        print(f"  Median:   ${final['median']:,.0f}")
        print(f"  Std Dev:  ${final['std']:,.0f}")
        print(f"  Min:      ${final['min']:,.0f}")
        print(f"  Max:      ${final['max']:,.0f}")
        print("-"*50)
        print("Percentile Projections:")
        print(f"  5th percentile:  ${final['percentile_5']:,.0f}")
        print(f"  95th percentile: ${final['percentile_95']:,.0f}")
        print("-"*50)
        print(f"Probability of Loss: {final['prob_loss']:.1f}%")
        print("="*50)

### Run Monte Carlo Simulation

Uncomment to debug.

In [ ]:
# Monte Carlo simulation - now passing weights!
mc_sim = MonteCarloSimulation(data, weights=weights, num_simulations=1000, time_horizon=252)
mc_sim.print_summary()
mc_sim.plot_simulation()

## Next Steps

In this tutorial, we've laid the groundwork for a powerful portfolio analysis tool. Moving forward, we plan to simplify the user interface to make it even more accessible for less technical users. We'll also be adding new features and building out the code base, including comprehensive unit testing to ensure reliability.

We're excited to invite members of the #FinTwit and finance community to contribute to this open-source project. Your insights and expertise can help us enhance this tool, making it a valuable resource for investors everywhere. Let's work together to create something truly impactful!

## Conclusion

This notebook will guide less technical investors through the process of loading financial data, calculating key performance metrics, visualizing both individual asset and total portfolio performance, and running Monte Carlo simulations. By following these steps, you can perform comprehensive portfolio analysis in Python. Feel free to reach out and share your feedback or submit pull requests for improvements!